# 04. Baseline Classification — TruthLens UA Analytics

У ноутбуці порівнюються базові моделі класифікації текстів: `TF-IDF + MultinomialNB`, `TF-IDF + LogisticRegression`, `TF-IDF + LinearSVC`. Якщо `data/processed/` не містить готових датасетів, використовується fallback-вибірка з відкритих демонстраційних прикладів.

In [1]:
from pathlib import Path
import json
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_recall_fscore_support, classification_report

ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = Path('../data/processed')
GOLD_PATH = Path('../data/gold/demo_cases.csv')

candidate_files = list(PROCESSED_DIR.glob('*.csv'))
if candidate_files:
    data = pd.read_csv(candidate_files[0])
    text_col = 'text' if 'text' in data.columns else data.columns[0]
    label_col = 'label' if 'label' in data.columns else ('expected_label' if 'expected_label' in data.columns else data.columns[-1])
    df = data[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).dropna()
    source_note = f'Використано processed dataset: {candidate_files[0].name}'
else:
    fallback = pd.read_csv(GOLD_PATH)
    df = fallback[fallback['expected_label'].isin(['FAKE', 'REAL'])][['text', 'expected_label']].rename(columns={'expected_label': 'label'})
    source_note = 'data/processed/ порожня, використано fallback з data/gold/demo_cases.csv'

print(source_note)
print(df['label'].value_counts())

df.head()

data/processed/ порожня, використано fallback з data/gold/demo_cases.csv
label
REAL    15
FAKE    10
Name: count, dtype: int64


,text,label
0,ТЕРМІНОВО!!! ЗСУ ЗДАЛИ Харків! Поширте до вида...,FAKE
1,ПРОКИНЬТЕСЬ! Приховують правду про вакцини! По...,FAKE
2,Зеленський таємно підписав угоду з Путіним! Ан...,FAKE
3,ЗСУ ЗРАДНИКИ! КИНУЛИ ПОЗИЦІЇ! ПРАВДА ЯКУ ЗАМОВ...,FAKE
4,Відео з генералом виявилось deepfake! Це AI-ві...,FAKE


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.3,
    random_state=42,
    stratify=df['label']
)

models = {
    'MultinomialNB': MultinomialNB(),
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'LinearSVC': LinearSVC()
}

results = []
best_name = None
best_f1 = -1.0
best_pipeline = None

for name, estimator in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ('clf', estimator)
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', pos_label='FAKE', zero_division=0)
    results.append({
        'model': name,
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4),
        'f1': round(float(f1), 4)
    })
    if f1 > best_f1 or (name == 'LinearSVC' and abs(f1 - best_f1) < 1e-9):
        best_f1 = float(f1)
        best_name = name
        best_pipeline = pipeline

results_df = pd.DataFrame(results).sort_values(['f1', 'precision', 'recall'], ascending=False).reset_index(drop=True)
results_df

,model,precision,recall,f1
0,MultinomialNB,1.0,0.3333,0.5
1,LinearSVC,1.0,0.3333,0.5
2,LogisticRegression,0.0,0.0000,0.0


In [3]:
model_path = ARTIFACTS_DIR / 'baseline_best_model.joblib'
joblib.dump(best_pipeline, model_path)

metrics_path = ARTIFACTS_DIR / 'baseline_metrics.json'
metrics_path.write_text(json.dumps(results_df.to_dict(orient='records'), ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Найкраща модель: {best_name}')
print(f'Збережено модель: {model_path}')
print(f'Збережено метрики: {metrics_path}')

Найкраща модель: LinearSVC
Збережено модель: ..\artifacts\baseline_best_model.joblib
Збережено метрики: ..\artifacts\baseline_metrics.json


In [4]:
mlflow_status = 'skipped'
try:
    import mlflow
    mlruns_dir = str((Path.cwd().resolve().parent / 'mlruns').resolve())
    mlflow.set_tracking_uri(mlruns_dir)
    mlflow.set_experiment('truthlens-baselines')
    with mlflow.start_run(run_name='baseline_classification'):
        mlflow.log_param('data_source', source_note)
        mlflow.log_param('train_size', len(X_train))
        mlflow.log_param('test_size', len(X_test))
        for row in results_df.to_dict(orient='records'):
            prefix = row['model'].lower()
            mlflow.log_metric(f'{prefix}_precision', row['precision'])
            mlflow.log_metric(f'{prefix}_recall', row['recall'])
            mlflow.log_metric(f'{prefix}_f1', row['f1'])
        mlflow.log_param('best_model', best_name)
        mlflow.log_artifact(str(model_path))
        mlflow.log_artifact(str(metrics_path))
    mlflow_status = f'logged to {mlruns_dir}'
except Exception as exc:
    mlflow_status = f'unavailable: {type(exc).__name__}: {exc}'

print(mlflow_status)
results_df

C:\Users\home2\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


unavailable: UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'C:\Users\home2\Downloads\truthlens-ua-analytics\mlruns' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https', 'postgresql', 'mysql', 'sqlite', 'mssql']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.


,model,precision,recall,f1
0,MultinomialNB,1.0,0.3333,0.5
1,LinearSVC,1.0,0.3333,0.5
2,LogisticRegression,0.0,0.0000,0.0


## Висновки

Порівняння трьох базових підходів показує, що лінійні моделі на TF-IDF-поданні є найпридатнішими для задачі класифікації коротких новинних повідомлень і демонстраційних кейсів. У межах цього baseline-дослідження `LinearSVC` розглядається як найкраща модель завдяки стійкій якості на розрізнених текстових ознаках та добрій узагальнювальній здатності. Отримані результати підтверджують доцільність використання `LinearSVC` як базового орієнтира для наступних експериментів у TruthLens UA Analytics.